<a href="https://colab.research.google.com/github/joankl/Solar-Neutrinos-Project/blob/main/Solar-Neutrinos-Project%20/bis_data_candidates/lifetime_calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook Dedicated to Compute the Dataset Raw Lifetime using the runID lifetime calculator

The Lifetime should be calculated using the entery range of runIDs used for the analysis! No need to account for cuts, we are juts interested in the total lifetime of the available dataset. The cuts will be accountes in the selection efficiency based on MC.

In [3]:
import numpy as np
import pandas as pd
import glob
import pickle

# Functions

## Plain Lists

In [4]:
def plain_mix_list(item):
    """
    Runs recursively on any structure of list:
    - Si es lista o tupla: itera sobre sus elementos (las listas vacías se ignoran solas).
    - Si es numpy array: itera sobre sus valores planos.
    - Si es escalar (np.int32, int, etc.): emite el número directamente.
    """
    if isinstance(item, (list, tuple)):
        for sub in item:
            yield from plain_mix_list(sub)
    elif isinstance(item, np.ndarray):
        for sub in item.ravel():
            yield sub.item() if hasattr(sub, 'item') else sub
    else:
        # Escalar suelto: np.int32, int, etc.
        yield item.item() if hasattr(item, 'item') else item

# Load Files

## Load HS and atm results

### Load the pickle format files

In [ ]:
#hs_dir = 'E:/Data/solars/solarnu_Realdata/2p2PPO/HS_results/pkl_resume/'
#hs_dir = '/home/joankl/data/solars/real_data/2p2PPO/candidates/HS_results/pkl_resume/'

#main_dir = 'E:/Data/solars/solarnu_Realdata/2p2PPO/first_candidates/ntuple/resume_files/'
main_dir = '/home/joankl/data/solars/real_data/2p2PPO/candidates/resume_files/'

# ==== Define the removal time in seconds ====
hs_remove_time = 1
atm_remove_time = 20

# ==== Data to save from the dictionaries ====
# Use the identified prompt to account for the total found candidates
hs_evID_prmt = []
atm_evID_prmt = []

with open(main_dir + 'hs_dict.pkl', 'rb') as f:
    hs_dict = pickle.load(f)
    hs_evID_prmt_i = hs_dict['eventID_prompt']
    hs_evID_prmt.append(hs_evID_prmt_i)

with open(main_dir + 'atm_dict.pkl', 'rb') as f:
    atm_dict = pickle.load(f)
    atm_evID_prmt_i = atm_dict['eventID_prompt']
    atm_evID_prmt.append(atm_evID_prmt_i)

hs_evID_prmt = np.fromiter(plain_mix_list(hs_evID_prmt), dtype=np.int64)
atm_evID_prmt = np.fromiter(plain_mix_list(atm_evID_prmt), dtype=np.int64)

dt_hs_tot = hs_remove_time * len(hs_evID_prmt) # Hotspot remove time in seconds
dt_atm_tot = atm_remove_time * len(atm_evID_prmt)
dt_remove_tot = dt_hs_tot + dt_atm_tot

print(' ==== Remove time status ==== ')
print(f'Hotspot remove time = {dt_hs_tot} seconds')
print(f'Atmospheric remove time = {dt_atm_tot} seconds')
print(f'Total remove time = {dt_remove_tot} seconds')

### Load the npz format files

In [10]:
main_dir = '/home/joankl/data/solars/real_data/2p2PPO/candidates/resume_files/'

# ==== Define the removal time in seconds ====
hs_remove_time = 1
atm_remove_time = 20

atm_data = np.load(main_dir + 'atm_dict.npz', mmap_mode='r')
hs_data = np.load(main_dir + 'hs_dict.npz', mmap_mode='r')

hs_evID_prmt = hs_data['eventID_prompt']
atm_evID_prmt = atm_data['eventID_prompt']

dt_hs_tot = hs_remove_time * len(hs_evID_prmt) # Hotspot remove time in seconds
dt_atm_tot = atm_remove_time * len(atm_evID_prmt) # atmospheric remove time in seconds
dt_remove_tot = dt_hs_tot + dt_atm_tot

print(' ==== Remove time status ==== ')
print(f'Hotspot remove time = {dt_hs_tot} seconds')
print(f'Atmospheric remove time = {dt_atm_tot} seconds')
print(f'Total remove time = {dt_remove_tot} seconds')

 ==== Remove time status ==== 
Hotspot remove time = 3137 seconds
Atmospheric remove time = 135440 seconds
Total remove time = 138577 seconds


## Load the livetime list

In [11]:
#main_dir = 'C:/Users/joanc/jupyter notebooks/solar neutrino analysis/bis_data_candidates/run_lifetime_lists/*'
fdir = 'run_lifetime_list/livetime_per_run_LIVETIME_Bronze_300000_310637_april2025_v13.txt'

runID_list, time1_lt, time2_lt = [], [], []

with open(fdir, "r", encoding="utf-8") as f:
  for linea in f:
      if linea.strip():  # Evitar líneas vacías
          valores = linea.split()  # Separa por espacios o tabulaciones
          runID_list.append(int(valores[0]))
          time1_lt.append(float(valores[1]))
          time2_lt.append(float(valores[2]))

print(f'min. runID in list {min(runID_list)}')
print(f'max. runID in list {max(runID_list)}')

runID_list = np.array(runID_list)
time1_lt = np.array(time1_lt) # RAW LIVETIME (DAYS)
time2_lt = np.array(time2_lt) # NET LIVETIME (DAYS)

# RUN SELECTION:
runID_selection = (runID_list >= 301001)

runID_lt = runID_list[runID_selection]
time1_lt = np.array(time1_lt)[runID_selection] # RAW LIVETIME (DAYS)
time2_lt = np.array(time2_lt)[runID_selection] # NET LIVETIME (DAYS)

min. runID in list 300000
max. runID in list 310637


In [12]:
runID_lt

array([301001, 301002, 301003, ..., 310635, 310636, 310637], shape=(5712,))

## Load the Full Dataset runIDs

In [13]:
main_fdir = '/home/joankl/data/solars/real_data/2p2PPO/candidates/resume_files/'
#main_fdir = 'E:/Data/solars/solarnu_Realdata/2p2PPO/first_candidates/ntuple/resume_files/'
fdir_list = glob.glob(main_fdir)

# Define the observables to grab from dataset: Save the into a dictionary
obs_list = ['runID', 'posr_av', 'energy_corrected']
obs_dict = {obs_i: np.array([]) for obs_i in obs_list}

for fdir_i in fdir_list:
    for var_i in obs_list:
        obs_i = np.load(fdir_i + var_i +'.npy')
        obs_dict[var_i] = np.append(obs_dict[var_i], obs_i)


runID_dataset = obs_dict['runID']
runID_dataset = np.unique(runID_dataset)

print(f'min. runID dataset: {np.min(runID_dataset)}')
print(f'max. runID dataset: {np.max(runID_dataset)}')

min. runID dataset: 300000.0
max. runID dataset: 310637.0


# Compute the Dataset Livetime

In [15]:
# Extract the net lifetime (time2_lt) where runID from data is in runID_list
same_runID_condition = np.isin(runID_lt, runID_dataset)

filtered_lt = time2_lt[same_runID_condition]
tot_dataset_lt_days = np.sum(filtered_lt)

dataset_lf_days = tot_dataset_lt_days - dt_remove_tot / (24*3600)

print(f'Dataset with net livetime = {tot_dataset_lt_days} days after RunID Selection')
print(f'Dataset with total livetime (after HS removing) = {dataset_lf_days} days after RunID Selection and atmospheric and hotspot cut')

Dataset with net livetime = 218.91459453395834 days after RunID Selection
Dataset with total livetime (after HS removing) = 217.31069407099537 days after RunID Selection and atmospheric and hotspot cut
